Embedding lookup: from tokens to vectors

How nn.Embedding works as a learnable lookup table with similarity computation

In [11]:
import torch
import torch.nn as nn
import torch.nn.functional as F

vocab_size = 10000 # number of unique tokens
embed_dim = 256 # embedding vector size

# creating embedding layer (stores a [10000,256] matrix)
embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)

print(f'Embedding matrix shape: {embedding.weight.shape}')
print(f'Total parameters: {embedding.weight.numel():,}')

# simulate the tokens for - the cat sat on the mat
token_ids = torch.tensor([42, 1587, 923, 15, 42, 2041])

# Lookup: each index selects a row from the matrix
vectors = embedding(token_ids)
print(f'\nInput Shape: {token_ids.shape}')
print(f'\nOutput Shape: {vectors.shape}')

# same token will get same vector always - eg "the" and "The"
print(f'Same token same vector: {torch.equal(vectors[0], vectors[4])}')

# why it works: equivalent to one hot x matrix
# one hot approach (wasteful but mathematically correct)

one_hot = F.one_hot(torch.tensor(42), num_classes = vocab_size).float()
manual_lookup = one_hot @ embedding.weight # matrix multiply

# direct indexing
direct_lookup = embedding.weight[42]

print(f'\nOne-hot multiply == direct index: '
      f'{torch.allclose(manual_lookup,direct_lookup)}')


# similarity: trained embeddings cluster similar words
# after training similar words have high cosine similarity
v1 = vectors[0] # the
v2 = vectors[1] # cat
cosine_sim = F.cosine_similarity(v1.unsqueeze(0), v2.unsqueeze(0))
print(f'\nCosine Similarity (untrained, random): {cosine_sim.item():.4f}')
print("After training, similar words would score > 0.7")

v4 = vectors[4] # cat
cosine_sim = F.cosine_similarity(v1.unsqueeze(0), v4.unsqueeze(0))
print(f'\nCosine Similarity: {cosine_sim.item():.4f}')
print("After training, similar words would score > 0.7")

Embedding matrix shape: torch.Size([10000, 256])
Total parameters: 2,560,000

Input Shape: torch.Size([6])

Output Shape: torch.Size([6, 256])
Same token same vector: True

One-hot multiply == direct index: True

Cosine Similarity (untrained, random): -0.0560
After training, similar words would score > 0.7

Cosine Similarity: 1.0000
After training, similar words would score > 0.7
